# Sales & Demand Forecasting for a UK Online Retailer

**Prepared for:** store owners, startup founders, and business managers who need a clear, no-jargon view of where sales are headed and how to plan around it.

**What this notebook does:** cleans a year of real transaction data, engineers time-based features (trend, seasonality, day-of-week effects), trains and evaluates forecasting models, and produces a 30-day forecast with business-facing charts and recommendations.

**Dataset:** [UCI Online Retail](https://archive.ics.uci.edu/ml/datasets/online+retail) — ~542,000 line-item transactions from a UK-based online retailer, December 2010 to December 2011.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_style import apply_style, currency_axis, SERIES_1, SERIES_2, SERIES_2_BAND

apply_style()
pd.set_option("display.max_columns", 20)


## 1. Load the raw data

In [2]:
raw = pd.read_excel("../data/raw/online_retail.xlsx")
print(f"Rows: {len(raw):,}  |  Columns: {list(raw.columns)}")
print(f"Date range: {raw['InvoiceDate'].min()} to {raw['InvoiceDate'].max()}")
raw.head()


Rows: 541,909  |  Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
raw.info()
print()
raw.describe(include="all").T


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB



,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909.0,25900.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Clean the transaction data

Raw retail transaction exports always need a pass of cleaning before they can be trusted:

- **Cancellations** — `InvoiceNo` values starting with `"C"` are returns/cancellations, not sales.
- **Bad quantities/prices** — a handful of rows have `Quantity <= 0` or `UnitPrice <= 0` (adjustments, data-entry errors); these aren't real sales either.
- **Missing product descriptions** — a small number of rows have no `Description`, usually bookkeeping adjustments rather than product sales.

`CustomerID` is missing for many rows, but that's irrelevant here — we're forecasting total revenue, not building a customer-level model, so we keep those rows.


In [4]:
n_before = len(raw)

df = raw.copy()
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df[df["Description"].notna()]
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

n_after = len(df)
pct_dropped = (n_before - n_after) / n_before * 100
print(f"Rows before cleaning: {n_before:,}")
print(f"Rows after cleaning:  {n_after:,}")
print(f"Dropped: {n_before - n_after:,} rows ({pct_dropped:.1f}%)")

assert (df["Revenue"] < 0).sum() == 0, "Negative revenue remains after cleaning"
assert df["Description"].isna().sum() == 0, "Missing descriptions remain after cleaning"
print("Checks passed: no negative revenue, no missing descriptions.")


Rows before cleaning: 541,909
Rows after cleaning:  530,104
Dropped: 11,805 rows (2.2%)
Checks passed: no negative revenue, no missing descriptions.


## 3. Build the daily revenue series

The task is to forecast **UK daily revenue** (about 91% of all transactions), since a single clear market tells a cleaner story than blending currencies and geographies. We also pull two descriptive-only summaries (top countries, top products) to support the business narrative later — those are not forecast individually.

The transaction data is at the line-item level, so we aggregate to one revenue total per calendar day, then fill in any day with zero recorded transactions as `0` (rather than silently dropping it) so the series has a regular daily frequency — required for lag/rolling features later. The final month (December 2011) is cut off mid-month in the source data, so we truncate the series at the last **complete** month to avoid the model mistaking a data cutoff for a demand collapse.


In [5]:
uk = df[df["Country"] == "United Kingdom"].copy()
uk["Date"] = uk["InvoiceDate"].dt.normalize()

daily = uk.groupby("Date")["Revenue"].sum()

full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_range, fill_value=0.0)
daily.index.name = "Date"

# Truncate the trailing partial month (Dec 2011 only has data through the 9th).
last_full_month_end = (daily.index.max().replace(day=1) - pd.Timedelta(days=1))
daily = daily.loc[:last_full_month_end]

assert len(daily) == (daily.index.max() - daily.index.min()).days + 1, "Gaps in the daily date index"
assert daily.isna().sum() == 0, "Unexpected NaNs in daily revenue series"

print(f"Daily series: {daily.index.min().date()} to {daily.index.max().date()}  ({len(daily)} days)")
print(f"Total revenue: £{daily.sum():,.0f}  |  Average day: £{daily.mean():,.0f}")

daily_revenue = daily.rename("Revenue").reset_index()
daily_revenue.to_csv("../data/processed/daily_revenue.csv", index=False)
daily_revenue.tail()


Daily series: 2010-12-01 to 2011-11-30  (365 days)
Total revenue: £8,432,600  |  Average day: £23,103


,Date,Revenue
360,2011-11-26,0.00
361,2011-11-27,19781.17
362,2011-11-28,52649.14
363,2011-11-29,65304.01
364,2011-11-30,50546.43


In [6]:
top_countries = (
    df.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(5)
)
top_products_uk = (
    uk.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)
)

print("Top 5 countries by revenue:")
print(top_countries.apply(lambda x: f"£{x:,.0f}"))
print()
print("Top 10 products by revenue (UK):")
print(top_products_uk.apply(lambda x: f"£{x:,.0f}"))


Top 5 countries by revenue:
Country
United Kingdom    £9,025,222
Netherlands         £285,446
EIRE                £283,454
Germany             £228,867
France              £209,715
Name: Revenue, dtype: object

Top 10 products by revenue (UK):
Description
DOTCOM POSTAGE                        £206,249
PAPER CRAFT , LITTLE BIRDIE           £168,470
REGENCY CAKESTAND 3 TIER              £142,273
WHITE HANGING HEART T-LIGHT HOLDER    £100,498
PARTY BUNTING                          £93,659
JUMBO BAG RED RETROSPOT                £86,471
MEDIUM CERAMIC TOP STORAGE JAR         £80,576
PAPER CHAIN KIT 50'S CHRISTMAS         £62,743
ASSORTED COLOUR BIRD ORNAMENT          £54,757
CHILLI LIGHTS                          £53,337
Name: Revenue, dtype: object


## 4. Time-based feature engineering

We don't have `statsmodels`/`prophet` available, so trend and seasonality are engineered as explicit columns a plain regression model can use directly:

- **Calendar features** — day of week, weekend flag, day of month, month, quarter, week of year, month start/end flags.
- **Cyclical encodings** — day-of-week and month expressed as `sin`/`cos` pairs, so the model sees "Sunday is next to Monday" and "December is next to January" instead of treating them as unrelated numbers.
- **Trend index** — a simple day-count since the series start, so a linear model can pick up on overall growth or decline.
- **Lag features** — revenue 1, 7, and 14 days ago (yesterday, same weekday last week, same weekday two weeks ago).
- **Rolling statistics** — 7- and 28-day rolling mean/std, computed on *shifted* data so today's row never sees today's own revenue (that would be leakage).
- **Holiday flag** — UK bank holidays and Christmas/Boxing Day/New Year, the single strongest seasonal driver in this dataset.

The first 28 rows don't have a full 28-day rolling window yet, so they're dropped before modeling.


In [7]:
import holidays as holidays_pkg

feat = daily_revenue.set_index("Date").copy()
feat.index = pd.DatetimeIndex(feat.index)

# Calendar features
feat["day_of_week"] = feat.index.dayofweek
feat["is_weekend"] = (feat["day_of_week"] >= 5).astype(int)
feat["day_of_month"] = feat.index.day
feat["month"] = feat.index.month
feat["quarter"] = feat.index.quarter
feat["week_of_year"] = feat.index.isocalendar().week.astype(int)
feat["is_month_start"] = feat.index.is_month_start.astype(int)
feat["is_month_end"] = feat.index.is_month_end.astype(int)

# Cyclical encodings
feat["dow_sin"] = np.sin(2 * np.pi * feat["day_of_week"] / 7)
feat["dow_cos"] = np.cos(2 * np.pi * feat["day_of_week"] / 7)
feat["month_sin"] = np.sin(2 * np.pi * feat["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * feat["month"] / 12)

# Trend
feat["trend_idx"] = np.arange(len(feat))

# Lag features (past actuals only)
feat["lag_1"] = feat["Revenue"].shift(1)
feat["lag_7"] = feat["Revenue"].shift(7)
feat["lag_14"] = feat["Revenue"].shift(14)

# Rolling stats computed on shifted data, so today's row never sees today's revenue
feat["rolling_mean_7"] = feat["Revenue"].shift(1).rolling(7).mean()
feat["rolling_mean_28"] = feat["Revenue"].shift(1).rolling(28).mean()
feat["rolling_std_7"] = feat["Revenue"].shift(1).rolling(7).std()

# Holiday flag (UK)
uk_holidays = holidays_pkg.UK(years=[2010, 2011])
feat["is_holiday"] = feat.index.to_series().apply(lambda d: 1 if d in uk_holidays else 0)

n_before_dropna = len(feat)
feat = feat.dropna()
print(f"Dropped {n_before_dropna - len(feat)} warm-up rows with incomplete rolling/lag windows.")
print(f"Feature table: {feat.shape[0]} rows x {feat.shape[1]} columns")
feat.head()


Dropped 28 warm-up rows with incomplete rolling/lag windows.
Feature table: 337 rows x 21 columns


,Revenue,day_of_week,is_weekend,day_of_month,month,quarter,week_of_year,is_month_start,is_month_end,dow_sin,...,month_sin,month_cos,trend_idx,lag_1,lag_7,lag_14,rolling_mean_7,rolling_mean_28,rolling_std_7,is_holiday
Date,,,,,,,,,,,,,,,,,,,,,
2010-12-29,0.0,2,0,29,12,4,52,0,0,0.974928,...,-2.449294e-16,1.000000,28,0.0,5488.85,30248.73,2269.080000,26723.892143,4125.861566,0
2010-12-30,0.0,3,0,30,12,4,52,0,0,0.433884,...,-2.449294e-16,1.000000,29,0.0,10394.71,47748.09,1484.958571,24766.103571,3928.831087,0
2010-12-31,0.0,4,0,31,12,4,52,0,1,-0.433884,...,-2.449294e-16,1.000000,30,0.0,0.00,34357.25,0.000000,23067.156071,0.000000,0
2011-01-01,0.0,5,1,1,1,1,52,1,0,-0.974928,...,5.000000e-01,0.866025,31,0.0,0.00,0.00,0.000000,21591.845714,0.000000,1
2011-01-02,0.0,6,1,2,1,1,52,0,0,-0.781831,...,5.000000e-01,0.866025,32,0.0,0.00,6653.94,0.000000,21591.845714,0.000000,0


## 5. Train / test split

The split must respect time — we never let the model train on the future to predict the past. We hold out the **last 56 days (8 weeks)** as a realistic test of "predict sales the model has never seen," and use a 5-fold `TimeSeriesSplit` on the training portion as a stability check (does performance hold up across different historical windows, not just one lucky split).


In [8]:
from sklearn.model_selection import TimeSeriesSplit

FEATURE_COLS = [
    "day_of_week", "is_weekend", "day_of_month", "month", "quarter", "week_of_year",
    "is_month_start", "is_month_end", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "trend_idx", "lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_mean_28",
    "rolling_std_7", "is_holiday",
]
TARGET_COL = "Revenue"

HOLDOUT_DAYS = 56
train_df = feat.iloc[:-HOLDOUT_DAYS]
test_df = feat.iloc[-HOLDOUT_DAYS:]

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

print(f"Train: {train_df.index.min().date()} to {train_df.index.max().date()}  ({len(train_df)} days)")
print(f"Test:  {test_df.index.min().date()} to {test_df.index.max().date()}  ({len(test_df)} days)")
assert train_df.index.max() < test_df.index.min(), "Train/test overlap detected"

tscv = TimeSeriesSplit(n_splits=5)
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    tr_range = (train_df.index[tr_idx.min()].date(), train_df.index[tr_idx.max()].date())
    val_range = (train_df.index[val_idx.min()].date(), train_df.index[val_idx.max()].date())
    print(f"Fold {i+1}: train {tr_range[0]}..{tr_range[1]}  ->  validate {val_range[0]}..{val_range[1]}")


Train: 2010-12-29 to 2011-10-05  (281 days)
Test:  2011-10-06 to 2011-11-30  (56 days)
Fold 1: train 2010-12-29..2011-02-17  ->  validate 2011-02-18..2011-04-04
Fold 2: train 2010-12-29..2011-04-04  ->  validate 2011-04-05..2011-05-20
Fold 3: train 2010-12-29..2011-05-20  ->  validate 2011-05-21..2011-07-05
Fold 4: train 2010-12-29..2011-07-05  ->  validate 2011-07-06..2011-08-20
Fold 5: train 2010-12-29..2011-08-20  ->  validate 2011-08-21..2011-10-05


## 6. Train forecasting models

Three models, in increasing sophistication:

1. **Naive baseline** ("predict same weekday last week") — not really a model, just the `lag_7` feature used directly as the prediction. Any real model that can't beat this isn't earning its complexity.
2. **Linear Regression** — interpretable; its coefficients on `trend_idx`, `month_sin/cos`, `dow_sin/cos` translate directly into plain-English statements later ("weekends run below average," "Q4 trend is rising").
3. **Random Forest** — captures nonlinear day-of-week x month interactions a linear model misses.


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# 1. Naive baseline: same weekday, one week ago
naive_pred = X_test["lag_7"].values

# 2. Linear Regression
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
lin_pred = lin_model.predict(X_test)

# 3. Random Forest
rf_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Models trained. Top features driving the Random Forest forecast:")
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
importances.head(8).apply(lambda x: f"{x:.1%}")


Models trained. Top features driving the Random Forest forecast:


lag_7              25.6%
trend_idx          14.1%
dow_sin            11.0%
day_of_week         9.2%
rolling_mean_28     7.2%
is_weekend          6.5%
rolling_mean_7      5.9%
lag_1               4.8%
dtype: object

## 7. Evaluate against the naive baseline

We score each model on the 56-day holdout using three metrics:

- **MAE** — average £ error per day (easiest to explain to a business owner)
- **RMSE** — same units, but penalizes big misses (spike days) more heavily
- **MAPE** — error as a % of actual revenue (computed only over days with `Revenue > 0`, to avoid dividing by zero on no-sales days)

The bar to clear is simple: **a real model must beat the naive "same day last week" guess.**


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(y_true, y_pred):
    mask = y_true > 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

results = []
for name, preds in [("Naive (lag-7)", naive_pred), ("Linear Regression", lin_pred), ("Random Forest", rf_pred)]:
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape_val = mape(y_test.values, preds)
    results.append({"Model": name, "MAE (£)": mae, "RMSE (£)": rmse, "MAPE (%)": mape_val})

results_df = pd.DataFrame(results).set_index("Model").round(1)
best_model_name = results_df["MAE (£)"].idxmin()
avg_daily_revenue = daily_revenue["Revenue"].mean()
best_mae = results_df.loc[best_model_name, "MAE (£)"]
best_mae_pct = best_mae / avg_daily_revenue * 100

print(results_df)
print()
beats_naive = best_mae < results_df.loc["Naive (lag-7)", "MAE (£)"]
print(f"Best model: {best_model_name}  (beats naive baseline: {beats_naive})")
print(
    f"On a typical day, the {best_model_name} forecast is off by about "
    f"£{best_mae:,.0f} — roughly {best_mae_pct:.1f}% of average daily revenue "
    f"(£{avg_daily_revenue:,.0f}/day)."
)


                   MAE (£)  RMSE (£)  MAPE (%)
Model                                         
Naive (lag-7)      11146.9   16276.2      32.6
Linear Regression  10480.3   14303.0      24.2
Random Forest      10695.4   15701.8      32.2

Best model: Linear Regression  (beats naive baseline: True)
On a typical day, the Linear Regression forecast is off by about £10,480 — roughly 45.4% of average daily revenue (£23,103/day).
